<p align="center">
    <span style="color:chocolate; font-size:2.5em; font-weight:bold;">
        eFleetPlan - Optimal infrastructure and fleet operation of electric LCV
    </span>
</p>

<p align="center">
    <span style="font-size:1.5em; font-weight:bold;">
        Carolina Gil Ribeiro, Jagruti Thaku
    </span>
</p>

## 0. Importing dependencies

In [30]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory 
import pandas as pd
import numpy as np
import xlsxwriter as xl
import matplotlib.pyplot as plt
import os
from glob import glob
from matplotlib import rcParams
from datetime import datetime
from pyomo.opt.results import SolverStatus
import datetime as dt
import math
from typing import Literal
import time
from matplotlib.ticker import MaxNLocator

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from optimalcharge_modules._1_schedule.schedule_generator_1 import ScheduleGenerator
from optimalcharge_modules._1_schedule.schedule_configure import scheduletype, vehicletype, companytype
from optimalcharge_modules._1_schedule.schedule_generator_2 import generate_schedules
from optimalcharge_modules._1_schedule.generate_graphs import generate_graphs



from optimalcharge_modules._2_optimisation.optimisation import assign, optimisation, save_results, calculate_days
from optimalcharge_modules._2_optimisation.optimisation_graphs import graph_vehicles, plot_summary_table, process_folder, graph_number_of_chargers_by_schedules, graph_chargingpower

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
notebook_dir = os.getcwd()

# <span style="color:chocolate;">2. Charging Infrastructure co-optimisation Module</span>

<span style="color:green">

## 2.1 optimisation parameters configurations

</span>

In [31]:
# environment optimisation arguments - adjust settings if necessary
opt_config = {# settings for the optimisation function              
              "opt_start_date": "2023-01-01 00:00:00",  # New schedule, start date
              "opt_end_date": "2023-01-31 23:00:00",  # New schedule, end date
              "freq": "h",  # 'h' means hourly frequency. .
              
              "EVs": 10, # Number of electric vehicles in the fleet
              
              #Input and output folders
              "output_folder" : os.path.join(project_root, 'data', 'Output', 'Schedule_5'),
              "input_folder" : os.path.join(project_root, 'data', 'Input'),
              
              # CSV file with electricity prices (the example is for SE3 in Stockholm, Sweden)
              "electricity_price_grid" : pd.read_csv(os.path.join(project_root, 'data', 'Input', 'SE3_el_prices_2023_modified.csv')),
              
              
              # Load the data for the vehicles created in the schedule generator module
              ## Energy consumption file
              "En_consumption" : pd.read_csv(os.path.join(project_root, 'data', 'Output', 'Schedule_5', '5_all_vehicles_consumption_km.csv')),
              ## Distance travel per vehicles and per hour file
              "Ev_distance" : pd.read_csv(os.path.join(project_root, 'data', 'Output', 'Schedule_5', '5_all_vehicles_Distance_km.csv')),
              ## Vehicles availability to charge at de distribution terminal              
              "Ev_availability_file" : pd.read_csv(os.path.join(project_root, 'data', 'Output', 'Schedule_5', '5_all_vehicles_ChargingStation.csv')),
              ## Vehicle battery power - important when the fleet has more than one type of vehicle                                         
              "Battery_Limitation" : pd.read_csv(os.path.join(project_root, 'data', 'Output', 'Schedule_5', '5_all_vehicles_PowerRating_kW.csv')) 
                                         
              }

## 2.2. Cost and power configuration

In the cost and power configurations above, it is possible to change the values of different paramenters, related to cost of infrastructure, energy subscription rates, route charging rates and battery power and losses.

In [32]:
cost_config = {"Infrastructure_life": 20, #lifetime or ownership time of the charging infrastucture
               "Infrastructure_cost": {'s': 32000 + 29000, # Slow charger cost
                                       50: 294000 + 472000, # Fast charger 50 kw cost
                                       150: 778000 + 644000, # Fast charger 150 kw cost
                                       350: 1451000 + 1202000}, # Fast charger 350 kw cost
               "Charger_Power" : {'s': 7.4, 'Route': 150, 50: 50, 150: 150, 350: 350},  # Charging power of each type of charger
               "maintenance_cost" : {'s': 5000, 50: 40000, 150: 120000, 350: 240000}, # Maintenance cost per type of charger
               
               "Infrastructure_subscription" : 50,  # SEK/day,
               "Price_FixedrateDT" : 0.0331,
               "Demand_rate" : 1352,
               "Price_FixedrateRoute" : 8.90,
               "Discount_rate": 0.05
               }

In [ ]:
power_config = {
    "Charging_losses": 0.964,  # Charging efficiency
    "Battery_Maximum Limit": 0.8,  # Maximum battery limit
    "Battery_Minimum Limit": 0.2,   # Minimum battery limit
    "Accumulated_Cycle_Capacity": 3500,  # Total energy throughput (in kWh) that the battery can handle
}

## 2.3 Run optimisation

In [34]:
# Call the optimisation function
m, Price, EV_availability, Distance_km = optimisation(opt_config, cost_config, power_config)

Number of days: 31
        1.63 seconds required to write file
        1.63 seconds required for presolve
Set parameter Username
Set parameter LicenseID to value 2678500
Academic license - for non-commercial use only - expires 2026-06-16
Read LP format model from file C:\Users\mcgr2\AppData\Local\Temp\tmpn0rt7a13.pyomo.lp
Reading time = 0.24 seconds
x1: 130924 rows, 126487 columns, 388086 nonzeros
Set parameter MIPGap to value 0.25
Set parameter ScaleFlag to value 2
Set parameter LogFile to value "gurobi_log_3.txt"
Set parameter Method to value -1
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) Ultra 7 165U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Non-default parameters:
MIPGap  0.25
ScaleFlag  2

Optimize a model with 130924 rows, 126487 columns and 388086 nonzeros
Model fingerprint: 0xd79bb51f
Variable types: 81857 continuous, 44630 integer (44630 bi

### Saving results

In [36]:
# Ensure the output directory exists before saving results
results_folder = os.path.join(project_root, 'data', 'Output', 'Schedule_5', 'Results')
os.makedirs(results_folder, exist_ok=True)

csv_file_pathA = os.path.join(results_folder, '5_Main_variables_results.csv')
csv_file_pathB = os.path.join(results_folder, '5_results_summary.csv')
csv_file_pathC = os.path.join(results_folder, '5_results_per_EV.csv')
csv_file_pathD = os.path.join(results_folder, '5_results_per_EV_descriptive.csv')

save_results(m, Price, EV_availability, Distance_km, csv_file_pathA, csv_file_pathB, csv_file_pathC, csv_file_pathD, power_config)

NameError: name 'power_config' is not defined

## 2.4. Post-processing and visualisation

In [ ]:
# Set the path to your folder
file_pattern_mean = os.path.join(project_root, 'data', 'Output', 'Schedule_5', 'Results', '*_results_summary.csv')

# Use glob to find the actual file
matched_files = glob(file_pattern_mean)
if not matched_files:
    raise FileNotFoundError(f"No file matches the pattern: {file_pattern_mean}")
file_path = matched_files[0]


plot_summary_table(file_path)

### Create maximum and average files

In [ ]:
# Set the path to your folder
folder_path = os.path.join(project_root, 'data', 'Output', 'Schedule_5', 'Results')

# Find all files with the pattern *_hourly_averages.csv
filename_pattern = os.path.join(folder_path, '*_Main_variables_results.csv')

process_folder(folder_path, filename_pattern)

In [ ]:
# Set the path to your folder
folder_path = os.path.join(project_root, 'data', 'Output', 'Schedule_5', 'Results')
file_pattern_max = os.path.join(project_root, 'data', 'Output', 'Schedule_5', 'Results', '*_max_variable_per_hour.csv')
files_hourly_max = glob(file_pattern_max)

graph_number_of_chargers_by_schedules(folder_path, files_hourly_max)

### Graph 2 - Average charging power and average price of electricity

In [ ]:
# Set the path to your folder
file_pattern_mean = os.path.join(project_root, 'data', 'Output', 'Schedule_5', 'Results', '*_avg_variable_per_hour.csv')

# Use glob to find the actual file
matched_files = glob(file_pattern_mean)
if not matched_files:
    raise FileNotFoundError(f"No file matches the pattern: {file_pattern_mean}")
file_path = matched_files[0]
folder_path = os.path.join(project_root, 'data', 'Output', 'Schedule_5', 'Results')

    
graph_chargingpower (file_path, folder_path)

### Graph 3 - Charging, discharging power and SOC

In [ ]:
# Set the path to your folder
file_pattern_pervehicle = os.path.join(project_root, 'data', 'Output', 'Schedule_5', 'Results', '*_results_per_EV_descriptive.csv')

# Use glob to find the actual file
matched_files = glob(file_pattern_pervehicle)
if not matched_files:
    raise FileNotFoundError(f"No file matches the pattern: {file_pattern_pervehicle}")
file_path = matched_files[0]


# Set the number of vehicles for the graph
n_vehicles = 5
# Set the number of days for the graph
n_days = 6


graph_vehicles(file_path,n_days, n_vehicles)